# 02 — Factor Construction

Build the daily factor panel used by the strategy: ATM IV30, IV rank, VXN excess rank, 2s10s slope rank. The original research panel built 77 factors; this notebook walks through the subset RGVH actually uses.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
%matplotlib inline

from src import factor_panel as fp

## 1. Build the panel

The whole pipeline (dedupe, IV-surface interpolation, rank computation) is wrapped by `factor_panel.build_factor_panel`.

In [ ]:
panel = fp.build_factor_panel(
    unified_path='../data/processed/spy_eod_unified.parquet',
    vix_path='../data/raw/vix_move.parquet',
    treasury_path='../data/raw/treasury_rates.parquet',
    output_path='../data/processed/factor_panel_daily.parquet',
)
panel.head(3)

## 2. ATM IV30 across the sample

In [ ]:
fig, ax = plt.subplots(figsize=(11,3))
panel.plot(x='tradeDate', y='F_iv_atm_30', ax=ax)
ax.set_title('SPY ATM 30-DTE IV across the sample')

## 3. Rolling 252-day rank (the actual signal)

Absolute IV is non-stationary. The rolling rank converts it to a regime-relative measure that is comparable across years.

In [ ]:
fig, ax = plt.subplots(figsize=(11,3))
panel.plot(x='tradeDate', y='F_iv_rank_252', ax=ax)
ax.axhline(0.70, color='red', ls='--', label='filter threshold ~ 0.70')
ax.set_ylim(0, 1); ax.legend(); ax.set_title('SPY F_iv_rank_252')

## 4. Yield curve and VXN excess (the macro / cross-asset features)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11,5), sharex=True)
panel.plot(x='tradeDate', y='slope_2s10s', ax=axes[0], color='black')
axes[0].axhline(0, color='red', ls='--', alpha=0.5)
axes[0].set_title('2s10s slope (10y - 2y)')
panel.plot(x='tradeDate', y='vxn_excess_rank_252', ax=axes[1], color='steelblue')
axes[1].axhline(0.75, color='red', ls='--', label='filter threshold')
axes[1].set_title('VXN-VIX rolling 252d rank')
axes[1].legend()

## 5. Correlations

The three filter inputs should not be too correlated, or they're not capturing different regimes.

In [ ]:
panel[['F_iv_rank_252','vxn_excess_rank_252','slope_2s10s_rank_252']].corr()